# NLP CW - Modeling

In this notebook we develop the PCL detection model.

## BestModel training is at the end of the notebook (section 6)

# 0. Imports and setup

In [1]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from urllib import request
import transformers, accelerate, torch
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)



In [2]:
# Set display options
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# Set random seed for reproducibility
def set_all_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

SEED = 42
set_all_seeds(SEED)

# 1. Data Loading

In [3]:
module_url = f"https://raw.githubusercontent.com/Perez-AlmendrosC/dontpatronizeme/master/semeval-2022/dont_patronize_me.py"
module_name = module_url.split('/')[-1]
print(f'Fetching {module_url}')
#with open("file_1.txt") as f1, open("file_2.txt") as f2
with request.urlopen(module_url) as f, open(module_name,'w') as outf:
  a = f.read()
  outf.write(a.decode('utf-8'))

from dont_patronize_me import DontPatronizeMe

dpm = DontPatronizeMe('.', "./task4_test.tsv")

dpm.load_task1()

train_ids = pd.read_csv('train_semeval_parids-labels.csv')
test_ids = pd.read_csv('dev_semeval_parids-labels.csv')

train_ids.par_id = train_ids.par_id.astype(str)
test_ids.par_id = test_ids.par_id.astype(str)

data=dpm.train_task1_df

data


Fetching https://raw.githubusercontent.com/Perez-AlmendrosC/dontpatronizeme/master/semeval-2022/dont_patronize_me.py


,par_id,art_id,keyword,country,text,label,orig_label
0,1,@@24942188,hopeless,ph,"We 're living in times of absolute insanity , ...",0,0
1,2,@@21968160,migrant,gh,"In Libya today , there are countless number of...",0,0
2,3,@@16584954,immigrant,ie,"""White House press secretary Sean Spicer said ...",0,0
3,4,@@7811231,disabled,nz,Council customers only signs would be displaye...,0,0
4,5,@@1494111,refugee,ca,""""""" Just like we received migrants fleeing El ...",0,0
...,...,...,...,...,...,...,...
10464,10465,@@14297363,women,lk,"""Sri Lankan norms and culture inhibit women fr...",0,1
10465,10466,@@70091353,vulnerable,ph,He added that the AFP will continue to bank on...,0,0
10466,10467,@@20282330,in-need,ng,""""""" She has one huge platform , and informatio...",1,3
10467,10468,@@16753236,hopeless,in,""""""" Anja Ringgren Loven I ca n't find a word t...",1,4


### Split the data to train and test (offical dev set) sets according to 'par_id' in CSV files

In [4]:
data_merged = data.copy()
data_merged["par_id"] = data_merged["par_id"].astype(str)

# Merge official split IDs with reconstructed dataset
train_val_df = train_ids.merge(data_merged, on="par_id", how="left", suffixes=("_csv", "_binary"))
# Use development set as my test set
test_df = test_ids.merge(data_merged, on="par_id", how="left", suffixes=("_csv", "_binary"))

train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.2,          # 20% validation
    random_state=42,
    stratify=train_val_df["label_binary"]
)
print("Train & Validation shape:", train_val_df.shape)
print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)


Train & Validation shape: (8375, 8)
Train shape: (6700, 8)
Validation shape: (1675, 8)
Test shape: (2094, 8)


### Display data for inspection

In [5]:
train_df


,par_id,label_csv,art_id,keyword,country,text,label_binary,orig_label
683,1393,"[1, 0, 0, 0, 0, 1, 0]",@@16413808,immigrant,my,"To me , I am always mindful that we are dealin...",1,4
2496,1903,"[0, 0, 0, 0, 0, 0, 0]",@@3872069,women,ca,"""As a result , Anderson quit her job and is no...",0,0
123,8276,"[1, 0, 1, 0, 0, 1, 0]",@@16736985,disabled,lk,When we talk about the freedom it is essential...,1,3
1424,702,"[0, 0, 0, 0, 0, 0, 0]",@@20477072,refugee,my,The report also indicated that the OIC members...,0,0
6665,6502,"[0, 0, 0, 0, 0, 0, 0]",@@22855617,disabled,gb,"When his mother , Mandy Pedelty , revealed the...",0,0
...,...,...,...,...,...,...,...,...
8324,8333,"[0, 0, 0, 0, 0, 0, 0]",@@3370471,refugee,ph,"The mother of 6 , including 3 adopted children...",0,0
549,9719,"[0, 0, 1, 1, 0, 1, 0]",@@1857896,hopeless,tz,"In many cases , resultant effects of such type...",1,3
1199,449,"[0, 0, 0, 0, 0, 0, 0]",@@319449,disabled,ie,HSE says it ca n't pay for services to help di...,0,0
2985,2437,"[0, 0, 0, 0, 0, 0, 0]",@@25567226,in-need,hk,A second T-Home project is being launched in t...,0,0


In [6]:
val_df

,par_id,label_csv,art_id,keyword,country,text,label_binary,orig_label
4094,3658,"[0, 0, 0, 0, 0, 0, 0]",@@17042937,refugee,ca,"""Viet Thanh Nguyen 's new book , """" The Refuge...",0,0
4637,4253,"[0, 0, 0, 0, 0, 0, 0]",@@4101327,in-need,in,"""Tewari said , """" Those people who are in the ...",0,0
1616,918,"[0, 0, 0, 0, 0, 0, 0]",@@21721527,disabled,gb,"""Kamran Mallick , chief executive of Disabilit...",0,0
3873,3420,"[0, 0, 0, 0, 0, 0, 0]",@@18630652,hopeless,gh,"If this is so , then why then do Africans blam...",0,0
3285,2765,"[0, 0, 0, 0, 0, 0, 0]",@@23426314,disabled,gh,My duties at CAF have never conflicted or disa...,0,0
...,...,...,...,...,...,...,...,...
2855,2292,"[0, 0, 0, 0, 0, 0, 0]",@@14849602,migrant,za,""""""" Illegal mining is largely fuelled by highl...",0,0
2100,1470,"[0, 0, 0, 0, 0, 0, 0]",@@22484933,vulnerable,ng,"""If Buhari were """" human , """" we would n't nee...",0,0
585,2344,"[1, 1, 0, 0, 1, 0, 0]",@@23526312,in-need,sg,"""Dr. C K Lee , Chief Executive and Medical Dir...",1,4
3831,3370,"[0, 0, 0, 0, 0, 0, 0]",@@1587662,vulnerable,ca,""""""" The number of incidents in this last time ...",0,1


In [7]:
test_df

,par_id,label_csv,art_id,keyword,country,text,label_binary,orig_label
0,4046,"[1, 0, 0, 1, 0, 0, 0]",@@14767805,hopeless,us,We also know that they can benefit by receivin...,1,3
1,1279,"[0, 1, 0, 0, 0, 0, 0]",@@7896098,refugee,ng,Pope Francis washed and kissed the feet of Mus...,1,4
2,8330,"[0, 0, 1, 0, 0, 0, 0]",@@17252299,refugee,ng,Many refugees do n't want to be resettled anyw...,1,2
3,4063,"[1, 0, 0, 1, 1, 1, 0]",@@3002894,in-need,ie,"""Budding chefs , like """" Fred """" , """" Winston ...",1,4
4,4089,"[1, 0, 0, 0, 0, 0, 0]",@@25597822,homeless,pk,"""In a 90-degree view of his constituency , one...",1,3
...,...,...,...,...,...,...,...,...
2089,10462,"[0, 0, 0, 0, 0, 0, 0]",@@22092971,homeless,gh,"The sad spectacle , which occurred on Saturday...",0,0
2090,10463,"[0, 0, 0, 0, 0, 0, 0]",@@4676355,refugee,pk,""""""" The Pakistani police came to our house and...",0,0
2091,10464,"[0, 0, 0, 0, 0, 0, 0]",@@19612634,disabled,ie,"""When Marie O'Donoghue went looking for a spec...",0,0
2092,10465,"[0, 0, 0, 0, 0, 0, 0]",@@14297363,women,lk,"""Sri Lankan norms and culture inhibit women fr...",0,1


# 2. Preprocessing

In [ ]:
import re
import html

# text cleaning for transformers
def clean_text_for_transformer(text):
    """
    Light text normalization for transformer-based classification.
    Preserves semantics and most punctuation while removing common artifacts.
    """
    if pd.isna(text):
        return ""
    text = str(text)

    # Unescape HTML entities (e.g., &amp; -> &, &quot; -> ")
    text = html.unescape(text)

    # Replace literal newline markers and real line breaks with spaces
    text = text.replace("\\n", " ")
    text = text.replace("\n", " ").replace("\r", " ").replace("\t", " ")

    # Normalize common unicode quotes/apostrophes/dashes
    replacements = {
        "\u2018": "'",   # left single quote
        "\u2019": "'",   # right single quote / apostrophe
        "\u201c": '"',   # left double quote
        "\u201d": '"',   # right double quote
        "\u2013": "-",   # en dash
        "\u2014": "-",   # em dash
        "\xa0": " ",     # non-breaking space
    }
    for src, tgt in replacements.items():
        text = text.replace(src, tgt)

    # fix spaced contractions
    # e.g., "do n ' t" -> "don't", "I ' m" -> "I'm"
    text = re.sub(r"\b([A-Za-z]+)\s+'\s+([A-Za-z]+)\b", r"\1'\2", text)
    text = re.sub(r"\b([A-Za-z])\s+'\s+([A-Za-z]+)\b", r"\1'\2", text)

    # Collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
# 1. Apply cleaning
train_df["text_clean"] = train_df["text"].apply(clean_text_for_transformer)
val_df["text_clean"]   = val_df["text"].apply(clean_text_for_transformer)
test_df["text_clean"]   = test_df["text"].apply(clean_text_for_transformer)

# 2. Keep only the necessary columns for modelling
keep_cols = ["par_id", "label_binary", "text_clean"]

cleaned_train_df = train_df[keep_cols].rename(columns={"label_binary": "label", "text_clean": "text"}).copy()
cleaned_train_df["label"] = cleaned_train_df["label"].astype(int)

cleaned_val_df = val_df[keep_cols].rename(columns={"label_binary": "label", "text_clean": "text"}).copy()
cleaned_val_df["label"] = cleaned_val_df["label"].astype(int)

cleaned_test_df = test_df[keep_cols].rename(columns={"label_binary": "label", "text_clean": "text"}).copy()
cleaned_test_df["label"] = cleaned_test_df["label"].astype(int)

print("Preprocessing applied.")
print("Train shape:", cleaned_train_df.shape)
print("Val shape:", cleaned_val_df.shape)
print("Test shape:", cleaned_test_df.shape)
display(cleaned_train_df.head())
print(cleaned_train_df["label"].value_counts(dropna=False))


Preprocessing applied.
Train shape: (6700, 3)
Val shape: (1675, 3)
Test shape: (2094, 3)


,par_id,label,text
683,1393,1,"To me , I am always mindful that we are dealin..."
2496,1903,0,"""As a result , Anderson quit her job and is no..."
123,8276,1,When we talk about the freedom it is essential...
1424,702,0,The report also indicated that the OIC members...
6665,6502,0,"When his mother , Mandy Pedelty , revealed the..."


label
0    6065
1     635
Name: count, dtype: int64


### Helper Functions



In [8]:
from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    # logits -> probs for positive class (label=1)
    probs = torch.softmax(torch.tensor(logits), dim=1)[:, 1].cpu().numpy()
    preds = (probs >= 0.5).astype(int)

    p, r, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": float(acc),
        "precision": float(p),
        "recall": float(r),
        "f1": float(f1),
    }

In [9]:
def tune_threshold_from_probs(y_true, y_prob):
    best = {
        "threshold": 0.5,
        "f1": -1.0,
        "precision": None,
        "recall": None,
        "accuracy": None,
    }

    for thr in np.linspace(0.05, 0.95, 91):
        y_hat = (y_prob >= thr).astype(int)
        f1 = f1_score(y_true, y_hat, average="binary", zero_division=0)

        if f1 > best["f1"]:
            best = {
                "threshold": float(thr),
                "f1": float(f1),
                "precision": float(precision_score(y_true, y_hat, zero_division=0)),
                "recall": float(recall_score(y_true, y_hat, zero_division=0)),
                "accuracy": float(accuracy_score(y_true, y_hat)),
            }

    return best

In [10]:
import torch.nn as nn

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels", inputs.get("label"))
        # Trainer may pass "label"; model expects "labels"
        if "label" in inputs and "labels" not in inputs:
            inputs = {k: v for k, v in inputs.items()}
            inputs["labels"] = inputs.pop("label")

        outputs = model(**inputs)
        logits = outputs.get("logits")

        cw = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss_fct = nn.CrossEntropyLoss(weight=cw)
        loss = loss_fct(logits.view(-1, model.config.num_labels), inputs["labels"].view(-1))

        return (loss, outputs) if return_outputs else loss

# 3. Modeling

In [ ]:
from datasets import Dataset

# Use Hugging Face Dataset objects
HF_train_df = Dataset.from_pandas(cleaned_train_df[["par_id", "text", "label"]].reset_index(drop=True))
HF_val_df = Dataset.from_pandas(cleaned_val_df[["par_id", "text", "label"]].reset_index(drop=True))
HF_test_df = Dataset.from_pandas(cleaned_test_df[["par_id", "text", "label"]].reset_index(drop=True))

print(HF_train_df)
print(HF_val_df)
print(HF_test_df)
print(HF_train_df[0])

Dataset({
    features: ['par_id', 'text', 'label'],
    num_rows: 6700
})
Dataset({
    features: ['par_id', 'text', 'label'],
    num_rows: 1675
})
Dataset({
    features: ['par_id', 'text', 'label'],
    num_rows: 2094
})
{'par_id': '1393', 'text': 'To me , I am always mindful that we are dealing with human beings with similar fears and hopes . I am of course not talking about the hardcore criminals but people like illegal immigrants , first time youthful offenders who may have slipped their way and such .', 'label': 1}


In [ ]:
# Tokenization

MODEL_NAME = "roberta-base"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

train_tok = HF_train_df.map(tokenize_batch, batched=True, batch_size=256)
val_tok = HF_val_df.map(tokenize_batch, batched=True, batch_size=256)
test_tok = HF_test_df.map(tokenize_batch, batched=True, batch_size=256)

# Set format for PyTorch
cols_for_torch = ["input_ids", "attention_mask", "label"]
train_tok.set_format(type="torch", columns=cols_for_torch)
val_tok.set_format(type="torch", columns=cols_for_torch)
test_tok.set_format(type="torch", columns=cols_for_torch)

print(train_tok)
print(val_tok)
print(test_tok)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/6700 [00:00<?, ? examples/s]

Map:   0%|          | 0/1675 [00:00<?, ? examples/s]

Map:   0%|          | 0/2094 [00:00<?, ? examples/s]

Dataset({
    features: ['par_id', 'text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 6700
})
Dataset({
    features: ['par_id', 'text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 1675
})
Dataset({
    features: ['par_id', 'text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 2094
})


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

print(model.config)

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaConfig {
  "add_cross_attention": false,
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 1,
  "tie_word_embeddings": true,
  "transformers_version": "5.0.0",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 50265
}



In [11]:
# Training arguments

OUTPUT_DIR = "./roberta_pcl"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # Core training
    learning_rate=1e-5,
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,

    # Evaluation
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    # Reproducibility
    seed=42,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

In [ ]:
# Trainer

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    #tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    class_weights=None,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Trainer ready.")

Trainer ready.


In [ ]:
# Train model
# Exp0: model trained without thresholding, weighted sampling, or oversampling

train_result = trainer.train()
print(train_result)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.263101,0.214801,0.919403,0.595238,0.471698,0.526316
2,0.187436,0.219241,0.914030,0.549669,0.522013,0.535484
3,0.167094,0.316586,0.918806,0.587786,0.484277,0.531034
4,0.099579,0.329375,0.920000,0.592593,0.503145,0.544218


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=1676, training_loss=0.1863399940345054, metrics={'train_runtime': 217.6615, 'train_samples_per_second': 123.127, 'train_steps_per_second': 7.7, 'total_flos': 3525688141824000.0, 'train_loss': 0.1863399940345054, 'epoch': 4.0})


# 4. Experiments

In [ ]:
# Exp1: test effect of threshold tuning

pred_out = trainer.predict(val_tok)
val_logits = pred_out.predictions
val_labels = pred_out.label_ids

val_probs = torch.softmax(torch.tensor(val_logits), dim=1)[:, 1].numpy()

thresholds = np.arange(0.05, 0.951, 0.01)

rows = []
for t in thresholds:
    preds = (val_probs >= t).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(val_labels, preds, average="binary", zero_division=0)
    acc = accuracy_score(val_labels, preds)
    rows.append({
        "threshold": round(float(t), 2),
        "accuracy": acc,
        "precision": p,
        "recall": r,
        "f1": f1
    })

thr_df = pd.DataFrame(rows).sort_values("f1", ascending=False).reset_index(drop=True)
display(thr_df.head(10))

best_row = thr_df.iloc[0]
best_threshold = float(best_row["threshold"])
print(f"Best threshold: {best_threshold:.2f}")
print(best_row)

,threshold,accuracy,precision,recall,f1
0,0.05,0.912239,0.533333,0.603774,0.566372
1,0.18,0.918209,0.570513,0.559748,0.565079
2,0.07,0.914030,0.543860,0.584906,0.563636
3,0.17,0.917612,0.566879,0.559748,0.563291
4,0.08,0.914627,0.547619,0.578616,0.562691
5,0.16,0.917015,0.563291,0.559748,0.561514
6,0.09,0.914627,0.548193,0.572327,0.560000
7,0.15,0.916418,0.559748,0.559748,0.559748
8,0.14,0.915821,0.556250,0.559748,0.557994
9,0.10,0.914627,0.548780,0.566038,0.557276


Best threshold: 0.05
threshold    0.050000
accuracy     0.912239
precision    0.533333
recall       0.603774
f1           0.566372
Name: 0, dtype: float64


In [ ]:
# Exp2: test effect of weighted sampling:

label_counts = cleaned_train_df["label"].value_counts().sort_index()
n_neg = int(label_counts[0])
n_pos = int(label_counts[1])
n_total = n_neg + n_pos

# Balanced class weights
# weight_c = N / (num_classes * count_c)
w_neg = n_total / (2 * n_neg)
w_pos = n_total / (2 * n_pos)

class_weights = torch.tensor([w_neg, w_pos], dtype=torch.float)

print("Label counts:", label_counts.to_dict())
print("Class weights:", class_weights.tolist())

Label counts: {0: 6065, 1: 635}
Class weights: [0.5523495674133301, 5.275590419769287]


In [ ]:
# Exp2 continued:

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    #tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    class_weights=class_weights, # modified to include weights
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
train_result = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.327870,0.861607,0.899701,0.479263,0.654088,0.553191
2,0.194740,1.948667,0.921194,0.690141,0.308176,0.426087
3,0.136972,1.530758,0.914627,0.551948,0.534591,0.543131


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

In [ ]:
# Exp3: Over Sampling PCL
OVERSAMPLE_FACTOR = 5

# copy training data
train_exp3 = cleaned_train_df.copy(deep=True)

# over sampling
maj_df = train_exp3[train_exp3["label"] == 0].copy()
min_df = train_exp3[train_exp3["label"] == 1].copy()
# Keep original minority once + add (factor-1)x sampled copies
extra_n = (OVERSAMPLE_FACTOR - 1) * len(min_df)
min_extra = min_df.sample(n=extra_n, replace=True, random_state=SEED) if extra_n > 0 else min_df.iloc[0:0].copy()
train_exp3 = pd.concat([maj_df, min_df, min_extra], axis=0).sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"Train label counts before over sampling:")
print(cleaned_train_df["label"].value_counts(dropna=False).sort_index())
print(f"Train label counts after over sampling:")
print(train_exp3["label"].value_counts(dropna=False).sort_index())

# Tokenization
HF_train_exp3 = Dataset.from_pandas(train_exp3[["par_id", "text", "label"]].reset_index(drop=True))
train_exp3_tok = HF_train_exp3.map(tokenize_batch, batched=True, batch_size=256)

print(HF_train_exp3)
print(train_exp3_tok)


Train label counts before over sampling:
label
0    6065
1     635
Name: count, dtype: int64
Train label counts after over sampling:
label
0    6065
1    3175
Name: count, dtype: int64


Map:   0%|          | 0/9240 [00:00<?, ? examples/s]

Dataset({
    features: ['par_id', 'text', 'label'],
    num_rows: 9240
})
Dataset({
    features: ['par_id', 'text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 9240
})


In [ ]:
#Exp 3 continued:

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_exp3_tok,
    eval_dataset=val_tok,
    #tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    class_weights=None,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
train_result = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.267697,0.325462,0.879403,0.424028,0.754717,0.542986
2,0.137898,0.417079,0.908657,0.515957,0.610063,0.559078
3,0.073343,0.621213,0.899701,0.480349,0.691824,0.567010
4,0.040547,0.566084,0.913433,0.543210,0.553459,0.548287


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

In [ ]:
"""pred_out = trainer.predict(test_tok)
test_logits = pred_out.predictions
test_labels = pred_out.label_ids

test_probs = torch.softmax(torch.tensor(test_logits), dim=1)[:, 1].numpy()
pred_label = (test_probs >= 0.5).astype(int)

y_true = pred_out.label_ids
f1 = f1_score(y_true, pred_label, zero_division=0)
print("F1:", f1)"""


F1: 0.5605095541401274
(2094,)


In [ ]:
# Exp 4: Hyperparameter sweep (Task 1 binary, dev F1)

import itertools

# sweep config
sweep_grid = {
    "learning_rate": [1e-5, 2e-5, 3e-5],
    "num_train_epochs": [3, 4],
    "weight_decay": [0.0, 0.01],
    "per_device_train_batch_size": [8, 16],
}

def run_one_experiment(exp_id, hp):

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2
    )

    # --- training args ---
    training_args = TrainingArguments(
        learning_rate=hp["learning_rate"],
        num_train_epochs=hp["num_train_epochs"],
        weight_decay=hp["weight_decay"],
        per_device_train_batch_size=hp["per_device_train_batch_size"],
        per_device_eval_batch_size=hp["per_device_train_batch_size"],
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_steps=100,
        report_to="none",
        fp16=torch.cuda.is_available(),
        seed=SEED,
        data_seed=SEED,
    )


    trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    #tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    class_weights=None,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
    train_result = trainer.train()

    # Return summary row
    row = {
        "exp_id": exp_id,
        "learning_rate": hp["learning_rate"],
        "num_train_epochs": hp["num_train_epochs"],
        "weight_decay": hp["weight_decay"],
        "batch_size": hp["per_device_train_batch_size"],

        # metrics @ 0.5
        "f1_at_05": float(eval_metrics.get("eval_f1", np.nan)),
        "precision_at_05": float(eval_metrics.get("eval_precision", np.nan)),
        "recall_at_05": float(eval_metrics.get("eval_recall", np.nan)),
        "accuracy_at_05": float(eval_metrics.get("eval_accuracy", np.nan)),
    }
    return row

# Build experiment list
keys = list(sweep_grid.keys())
values = [sweep_grid[k] for k in keys]
combos = [dict(zip(keys, v)) for v in itertools.product(*values)]

print(f"Total experiments: {len(combos)}")
display(pd.DataFrame(combos))

# Run sweep
all_rows = []
for i, hp in enumerate(combos, start=1):
    exp_id = i
    print("\n" + "="*80)
    print(f"Running {i}/{len(combos)}: {exp_id}")
    print(hp)
    print("="*80)

    try:
        row = run_one_experiment(exp_id, hp)
        all_rows.append(row)

        # Save/update cumulative results after each run
        results_df = pd.DataFrame(all_rows).sort_values("f1_tuned", ascending=False).reset_index(drop=True)

        print("Done:", exp_id)
        print("F1@0.5 =", row["f1_at_05"], "| F1 tuned =", row["f1_tuned"], "| best thr =", row["best_threshold"])
        display(results_df.head(10))

    except Exception as e:
        print(f"FAILED: {exp_id}")
        print(repr(e))
        all_rows.append({
            "exp_id": exp_id,
            "learning_rate": hp["learning_rate"],
            "num_train_epochs": hp["num_train_epochs"],
            "weight_decay": hp["weight_decay"],
            "error": repr(e),
        })

Total experiments: 24


,learning_rate,num_train_epochs,weight_decay,per_device_train_batch_size
0,0.00001,3,0.00,8
1,0.00001,3,0.00,16
2,0.00001,3,0.01,8
3,0.00001,3,0.01,16
4,0.00001,4,0.00,8
5,0.00001,4,0.00,16
6,0.00001,4,0.01,8
7,0.00001,4,0.01,16
8,0.00002,3,0.00,8
9,0.00002,3,0.00,16



Running 1/24: 1
{'learning_rate': 1e-05, 'num_train_epochs': 3, 'weight_decay': 0.0, 'per_device_train_batch_size': 8}


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

FAILED: 24
NameError("name 'eval_metrics' is not defined")


### Exp 5: We test the model with no preprocessing, we use this as a baseline for comparison with BestModel in Exercise 5.

In [12]:
#Exp 5: No Preprocessing

from datasets import Dataset

# copy data and Keep only the necessary columns for modelling
keep_cols = ["par_id", "label_binary", "text"]

train_exp5 = train_df[keep_cols].rename(columns={"label_binary": "label"}).copy()
train_exp5["label"] = train_exp5["label"].astype(int)

val_exp5 = val_df[keep_cols].rename(columns={"label_binary": "label"}).copy()
val_exp5["label"] = val_exp5["label"].astype(int)

test_exp5 = test_df[keep_cols].rename(columns={"label_binary": "label"}).copy()
test_exp5["label"] = test_exp5["label"].astype(int)

HF_train_exp5 = Dataset.from_pandas(train_exp5[["par_id", "text", "label"]].reset_index(drop=True))
HF_val_exp5 = Dataset.from_pandas(val_exp5[["par_id", "text", "label"]].reset_index(drop=True))
HF_test_exp5 = Dataset.from_pandas(test_exp5[["par_id", "text", "label"]].reset_index(drop=True))

# tokenization setup
MODEL_NAME = "roberta-base"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

# tokenize
train_exp5_tok = HF_train_exp5.map(tokenize_batch, batched=True, batch_size=256)
val_exp5_tok = HF_val_exp5.map(tokenize_batch, batched=True, batch_size=256)
test_exp5_tok = HF_test_exp5.map(tokenize_batch, batched=True, batch_size=256)

# Set format for PyTorch
cols_for_torch = ["input_ids", "attention_mask", "label"]
train_exp5_tok.set_format(type="torch", columns=cols_for_torch)
val_exp5_tok.set_format(type="torch", columns=cols_for_torch)
test_exp5_tok.set_format(type="torch", columns=cols_for_torch)

print(train_exp5_tok)
print(val_exp5_tok)
print(test_exp5_tok)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/6700 [00:00<?, ? examples/s]

Map:   0%|          | 0/1675 [00:00<?, ? examples/s]

Map:   0%|          | 0/2094 [00:00<?, ? examples/s]

Dataset({
    features: ['par_id', 'text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 6700
})
Dataset({
    features: ['par_id', 'text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 1675
})
Dataset({
    features: ['par_id', 'text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 2094
})


In [15]:
# Exp 5 continued

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_exp5_tok,
    eval_dataset=val_exp5_tok,
    #tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    class_weights=None,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
train_result = trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.256039,0.212504,0.918806,0.586466,0.490566,0.534247
2,0.187589,0.224812,0.915224,0.551515,0.572327,0.561728
3,0.156722,0.304391,0.918806,0.578231,0.534591,0.555556
4,0.097853,0.331279,0.919403,0.585714,0.515723,0.548495


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

In [16]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [18]:
# Extract the binary 0/1 labels for the official dev set (my test set) to a file
exp5_pred_out = trainer.predict(test_exp5_tok)
exp5_test_logits = exp5_pred_out.predictions
exp5_test_labels = exp5_pred_out.label_ids

exp5_test_probs = torch.softmax(torch.tensor(exp5_test_logits), dim=1)[:, 1].numpy()
exp5_pred_label = (exp5_test_probs >= 0.5).astype(int)

with open("exp5.txt", "w") as f:
    for p in exp5_pred_label:
        f.write(f"{int(p)}\n")
# Save it
!cp exp5.txt /content/drive/MyDrive/

# 6. BestModel

### Now train the Bst model, we do this by combining the train and validation set into one training set and then use best hyberparameters, Finally we test on the test set (offical dev set)

In [ ]:
from datasets import concatenate_datasets

# Concatenate training and validation sets:
HF_trainval = concatenate_datasets([HF_train_df, HF_val_df]).shuffle(seed=42)

# Tokenization after concatenation:
trainval_tok = HF_trainval.map(tokenize_batch, batched=True, batch_size=256)

# Set format for PyTorch
cols_for_torch = ["input_ids", "attention_mask", "label"]
trainval_tok.set_format(type="torch", columns=cols_for_torch)

print(trainval_tok)

Map:   0%|          | 0/8375 [00:00<?, ? examples/s]

Dataset({
    features: ['par_id', 'text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 8375
})


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
set_all_seeds(42)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # Core training
    learning_rate=1e-5,
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,

    # Evaluation
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=1,

    # Reproducibility
    seed=42,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

# Train with best hyperparameters, no thresholding, oversampling or weighted sampling
trainer = WeightedTrainer(
    model=model,
    args=training_args, # best training args from above experements
    train_dataset=trainval_tok, #train on the cobined data set
    eval_dataset=test_tok, # test on the offical dev set
    #tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    class_weights=None,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
trainer.train()
trainer.save_model("BestModel/model")  # saves model weights + config
tokenizer.save_pretrained("BestModel/model")
!cp -r BestModel /content/drive/MyDrive/BestModel

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.244669,0.199099,0.919293,0.589286,0.497487,0.539510
2,0.189458,0.227163,0.917861,0.582822,0.477387,0.524862
3,0.160531,0.268838,0.926934,0.640244,0.527638,0.578512
4,0.125929,0.303679,0.925024,0.623529,0.532663,0.574526


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# Extract the binary 0/1 labels for the official dev set (my test set) to a file
pred_out = trainer.predict(test_tok)
test_logits = pred_out.predictions
test_labels = pred_out.label_ids

test_probs = torch.softmax(torch.tensor(test_logits), dim=1)[:, 1].numpy()
pred_label = (test_probs >= 0.5).astype(int)

with open("dev.txt", "w") as f:
    for p in pred_label:
        f.write(f"{int(p)}\n")
# Save it
!cp dev.txt /content/drive/MyDrive/

### prepare the unlabeled test data and extract the predictions

In [ ]:
# load data
dpm.load_test()
unlabeled_test_data=dpm.test_set_df
unlabeled_test_data


,par_id,art_id,keyword,country,text
0,t_0,@@7258997,vulnerable,us,"In the meantime , conservatives are working to..."
1,t_1,@@16397324,women,pk,In most poor households with no education chil...
2,t_2,@@16257812,migrant,ca,The real question is not whether immigration i...
3,t_3,@@3509652,migrant,gb,"In total , the country 's immigrant population..."
4,t_4,@@477506,vulnerable,ca,"Members of the church , which is part of Ken C..."
...,...,...,...,...,...
3827,t_3893,@@20319448,migrant,jm,In a letter dated Thursday to European Commiss...
3828,t_3894,@@9990672,poor-families,au,They discovered that poor families with health...
3829,t_3895,@@37984,migrant,ca,"She married at 19 , to Milan ( Emil ) Badovina..."
3830,t_3896,@@9691377,immigrant,us,The United Kingdom is n't going to devolve int...


In [ ]:
# preprocess the test and tokenize it

# Apply cleaning
unlabeled_test_data["text_clean"] = unlabeled_test_data["text"].apply(clean_text_for_transformer)

# Keep only the necessary columns for modelling
keep_cols = ["par_id", "text_clean"]
cleaned_unlabeled_test = unlabeled_test_data[keep_cols].rename(columns={"text_clean": "text"}).copy()

print("Preprocessing applied.")
print("Test shape:", cleaned_unlabeled_test.shape)
display(cleaned_unlabeled_test.head())

# Use Hugging Face Dataset objects
HF_unlabeled_test = Dataset.from_pandas(cleaned_unlabeled_test[["par_id", "text"]].reset_index(drop=True))
print(HF_unlabeled_test)

# tokenize
unlabeled_test_tok = HF_unlabeled_test.map(tokenize_batch, batched=True, batch_size=256)

# Set format for PyTorch
cols_for_torch = ["input_ids", "attention_mask"]
unlabeled_test_tok.set_format(type="torch", columns=cols_for_torch)
print(unlabeled_test_tok)

Preprocessing applied.
Test shape: (3832, 2)


,par_id,text
0,t_0,"In the meantime , conservatives are working to..."
1,t_1,In most poor households with no education chil...
2,t_2,The real question is not whether immigration i...
3,t_3,"In total , the country 's immigrant population..."
4,t_4,"Members of the church , which is part of Ken C..."


Dataset({
    features: ['par_id', 'text'],
    num_rows: 3832
})


Map:   0%|          | 0/3832 [00:00<?, ? examples/s]

Dataset({
    features: ['par_id', 'text', 'input_ids', 'attention_mask'],
    num_rows: 3832
})


In [ ]:
# Extract the binary 0/1 labels for the unlabeled test set to a file
pred_test = trainer.predict(unlabeled_test_tok)
unlabeled_test_logits = pred_test.predictions

unlabeled_test_probs = torch.softmax(torch.tensor(unlabeled_test_logits), dim=1)[:, 1].numpy()
label = (unlabeled_test_probs >= 0.5).astype(int)

with open("test.txt", "w") as f:
    for l in label:
        f.write(f"{int(l)}\n")
# Save it
!cp test.txt /content/drive/MyDrive/